# 03 · Generalization  —  implementation-invariance & multipredicate transfer

- **E1** implementation-invariance — fix the done-*semantics* (item b done), vary its surface code; does the pivot query cluster by *what's done*, not *how written*?
- **E2** multipredicate transfer — done-set × generated-frontier confusion matrix via query transplant; and the pivot-1-quiet / pivot-2-active signature (nothing to subtract at the first pivot).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("..")  # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")  # loads model + head masks (once)
from _diff_primitives import *  # helper functions into scope

mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = (
    P.GOAL_HEADS,
    P.BOUNDARY_HEADS,
    P.UNION_HEADS,
    P.GROUP,
)

2026-08-31 17:31:51 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# ── multipredicate example: 4 INDEPENDENT computations, one assignment line each ──
# All orderings valid — the pivot choice is a genuine "which sub-task next", not a data dependency.
# Item `a` is the ANCHOR: ALWAYS prefilled — one in-context example that fixes the coding style
# (an inline list-comprehension). Progress analysis focuses on the remaining {b, c, d}.

USE_NATURAL_NAMES = False        # False → slot names a/b/c/d ; True → descriptive variable names
ITEMS_CONFIG      = "words_C"    # ← pick a preset from ITEMS_PRESETS below

# each preset: slot -> (condition text, RHS expression, natural variable name).
# convention: slot `a` is always a list-comprehension filter (the anchor / style example).
# all presets share the domain  def func(words: list[str])  so build_prefill/QUESTION stay valid.
ITEMS_PRESETS = {
    "words_A": {
        "a": ("the list of words that start with 'a'",          "[w for w in words if w.startswith('a')]",           "a_words"),
        "b": ("the count of palindromic words",                 "sum(w == w[::-1] for w in words)",                  "palindrome_count"),
        "c": ("the length of the longest word",                 "max(len(w) for w in words)",                        "longest_len"),
        "d": ("the list of words with more than five letters",  "[w for w in words if len(w) > 5]",                  "long_words"),
    },
    "words_B": {
        "a": ("the list of words that contain the letter 'e'",  "[w for w in words if 'e' in w]",                    "words_with_e"),
        "b": ("the total number of characters across all words","sum(len(w) for w in words)",                        "total_chars"),
        "c": ("the number of distinct words",                   "len(set(words))",                                   "distinct_count"),
        "d": ("the list of words sorted by length",             "sorted(words, key=len)",                            "by_length"),
    },
    "words_C": {
        "a": ("the list of words that end with 's'",            "[w for w in words if w.endswith('s')]",             "plural_words"),
        "b": ("the average word length",                        "sum(len(w) for w in words) / len(words)",           "avg_len"),
        "c": ("the number of words containing a digit",         "sum(any(ch.isdigit() for ch in w) for w in words)", "digit_word_count"),
        "d": ("the list of capitalized words",                  "[w.capitalize() for w in words]",                   "capitalized"),
    },
}
ITEMS   = ITEMS_PRESETS[ITEMS_CONFIG]
ORDER   = ["a", "b", "c", "d"]
ANCHOR  = "a"                    # always prefilled, never a pivot choice
ANALYZE = ["b", "c", "d"]       # the progress set we sweep / inject over

def varname(k): return ITEMS[k][2] if USE_NATURAL_NAMES else k
def assign(k):  return f"{varname(k)} = {ITEMS[k][1]}"

# conditions only — no dictated variable names; the LM names what it writes at/after the pivot
TASK_SPEC = "\n".join(f"({i+1}) {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill = ANCHOR (always) + assignments in `done` (⊆ ANALYZE), kept in ORDER; pivot at next indent."""
    keep = [k for k in ORDER if k == ANCHOR or k in done]
    lines = ["<code>", "def func(words: list[str]):"] + [f"    {assign(k)}" for k in keep]
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are identical across done-sets (task precedes prefill) — compute once.
_p0 = build_prompt([])                        # {} means "anchor a only"
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={a} (anchor only)", _p0,               ITEM_SPAN["b"])
show_prompt("done={a,b}           ", build_prompt(["b"]), ITEM_SPAN["c"])
print(f"config={ITEMS_CONFIG}  item spans:", ITEM_SPAN, " pivot({a})=", pivot_token(_p0))

╭─ done={a} (anchor only)  TASK span (79, 83)  ·  pivot=['    ']  ·  L=135 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ done={a,b}             TASK span (87, 94)  ·  pivot=['    ']  ·  L=151 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

config=words_C  item spans: {'a': (66, 76), 'b': (79, 83), 'c': (87, 94), 'd': (98, 103)}  pivot({a})= ['    ']


## E1 · Implementation-invariance of the query

Anchor `a` is always prefilled. Item **b** is done in three surface forms (generator / list-len / explicit
loop); frontier ∈ {c,d} in all. Cosine (over GOAL q-head slices) between `Δq = q(variant) − q({a})` across
variants.
**Predict:** high pairwise cosine ⇒ the "PROGRESS" the query encodes is the semantic done-set, not tokens.

In [3]:
# E1: fix the done-SEMANTICS (item b done), vary its SURFACE code; anchor a always prefilled.
VB = varname("b")
# three surface forms of item b's computation, per preset (must compute the same thing as ITEMS[b]).
_SURFACE = {
    "words_A": {  # count of palindromic words
        "generator": f"{VB} = sum(w == w[::-1] for w in words)",
        "listlen":   f"{VB} = len([w for w in words if w == w[::-1]])",
        "loop":      f"{VB} = 0\n    for w in words:\n        if w == w[::-1]:\n            {VB} += 1",
    },
    "words_B": {  # total number of characters
        "generator": f"{VB} = sum(len(w) for w in words)",
        "joinlen":   f"{VB} = len(''.join(words))",
        "loop":      f"{VB} = 0\n    for w in words:\n        {VB} += len(w)",
    },
    "words_C": {  # average word length
        "comprehension": f"{VB} = sum(len(w) for w in words) / len(words)",
        "map":           f"{VB} = sum(map(len, words)) / len(words)",
        "loop":          f"total = 0\n    for w in words:\n        total += len(w)\n    {VB} = total / len(words)",
    },
}
B_VARIANTS = _SURFACE.get(ITEMS_CONFIG, {"canonical": assign("b")})   # fallback: config's own single impl
qheads = heads_per_layer(GOAL_HEADS, "q"); Q_LAYERS = sorted(qheads); hd = mt.head_dim
p0 = build_prompt([]); q0 = cache_src_proj(p0, [pivot_of(p0)], Q_LAYERS, "q")   # baseline = anchor {a}
rich_print(Panel(Text.from_markup(exact_prompt(p0, patch_pos=[pivot_of(p0)])),
                 title=f"[bold]E1 · baseline p0 = {{a}} · cache @ pivot {pivot_of(p0)} = "
                       f"{pivot_token(p0, patch_pos=[pivot_of(p0)])!r}[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
_anchor = f"    {assign('a')}\n"

def goal_q_vec(qcache):
    return torch.cat([qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache])

dqs = {}
for name, impl in B_VARIANTS.items():
    p = get_prompt(QUESTION, "<code>\ndef func(words: list[str]):\n" + _anchor + "    " + impl + "\n    ")
    q = cache_src_proj(p, [pivot_of(p)], Q_LAYERS, "q")
    rich_print(Panel(Text.from_markup(exact_prompt(p, patch_pos=[pivot_of(p)])),
                     title=f"[bold]E1 · variant {name!r} (item b) · cache @ pivot {pivot_of(p)} = "
                           f"{pivot_token(p, patch_pos=[pivot_of(p)])!r}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))
    dqs[name] = goal_q_vec(q_lincomb([(1.0, q), (-1.0, q0)]))

names = list(B_VARIANTS)
tbl = Table(title=f"E1 · cos(Δq) across surface implementations of item b ({ITEMS_CONFIG})", highlight=False)
tbl.add_column("")
for n in names: tbl.add_column(n, justify="right")
for n1 in names:
    tbl.add_row(n1, *[f"{cos(dqs[n1], dqs[n2]):+.3f}" for n2 in names])
rich_print(tbl)
rich_print("[grey58]off-diagonal ≈ 1 ⇒ query invariant to surface implementation (clusters by semantics)[/grey58]")

╭─ E1 · baseline p0 = {a} · cache @ pivot 134 = ['    '] ─────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ E1 · variant 'comprehension' (item b) · cache @ pivot 150 = ['    '] ──────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ E1 · variant 'map' (item b) · cache @ pivot 147 = ['    '] ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(map(len, words)) / len(words)\n                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ E1 · variant 'loop' (item b) · cache @ pivot 160 = ['    '] ───────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     total = 0\n                                                                                                 │
│     for w in words:\n                                                                                           │
│         total += len(w)\n                                                                                       │
│     b = total / len(words)\n                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

E1 · cos(Δq) across surface implementations of item
                    b (words_C)                    
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┓
┃               ┃ comprehension ┃    map ┃   loop ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━┩
│ comprehension │        +1.000 │ +0.979 │ +0.921 │
│ map           │        +0.979 │ +1.000 │ +0.889 │
│ loop          │        +0.921 │ +0.889 │ +1.000 │
└───────────────┴───────────────┴────────┴────────┘

off-diagonal ≈ 1 ⇒ query invariant to surface implementation (clusters by semantics)

## E2 · Multipredicate transfer — done-set × frontier confusion matrix

Anchor `a` is always prefilled. Cache the pivot query for each source done-set over {b,c,d}, transplant
into the `{a}`-baseline dest, and record which variable the model writes next. **Predict:** the frontier =
one of the *un*done items (done {a,b} → c/d; done {a,b,c} → d). This is the Spec-1 query transplant with
clean combinatorics.

In [4]:
def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(GOAL_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt); piv0 = pivot_of(p0)   # dest = anchor {a}
rich_print(Panel(Text.from_markup(exact_prompt(p0, patch_pos=[piv0])),
                 title=f"[bold]E2 · DEST p0 = {{a}} · inject @ pivot {piv0} = "
                       f"{pivot_token(p0, patch_pos=[piv0])!r}[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
SRC_DONE = [["b"], ["c"], ["d"], ["b", "c"], ["b", "d"], ["c", "d"]]
rich_print(f"[grey70]clean {{a}} (no transplant):[/grey70] {escape(repr(greedy(dest_tok, max_new=20)))}")
for done in SRC_DONE:
    p = build_prompt(done)
    q = cache_src_proj(p, [pivot_of(p)], sorted(heads_per_layer(GOAL_HEADS, 'q')), "q")
    rich_print(Panel(Text.from_markup(exact_prompt(p, patch_pos=[pivot_of(p)])),
                     title=f"[bold]E2 · SRC done={{{','.join([ANCHOR] + done)}}} · cache @ pivot "
                           f"{pivot_of(p)} = {pivot_token(p, patch_pos=[pivot_of(p)])!r}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))
    pf, dc = q_inject_specs(q, piv0)
    g = greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=20, max_new=20)
    rich_print(f"src done={{{','.join([ANCHOR] + done):7s}}} → dest writes: [italic]{escape(repr(g))}[/italic]")

╭─ E2 · DEST p0 = {a} · inject @ pivot 134 = ['    '] ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean {a} (no transplant): ' b = sum(len(w) for w in words) / len(words) if words else 0'

╭─ E2 · SRC done={a,b} · cache @ pivot 150 = ['    '] ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

src done={a,b    } → dest writes: ' b = sum(any(c.isdigit() for c in w) for w in words)\n    c ='

╭─ E2 · SRC done={a,c} · cache @ pivot 152 = ['    '] ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

src done={a,c    } → dest writes: ' b = sum(len(w) for w in words) / len(words) if words else 0'

... (4 outputs left out)

╭─ E2 · SRC done={a,b,d} · cache @ pivot 162 = ['    '] ──────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│     d = [w.capitalize() for w in words]\n                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

src done={a,b,d  } → dest writes: ' b = sum(1 for w in words if any(c.isdigit() for c in w))\n   '

╭─ E2 · SRC done={a,c,d} · cache @ pivot 164 = ['    '] ──────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│     d = [w.capitalize() for w in words]\n                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

src done={a,c,d  } → dest writes: ' b = sum(len(w) for w in words) / len(words) if words else 0'

## E2b · Pivot-1-quiet / pivot-2-active signature

At the **first choice-pivot** (baseline `{a}`, nothing among {b,c,d} done) there is no PROGRESS among the
analyzed items to subtract — which item to do first is a free priority, not a subtraction. **Predict:** the
goal-head write is weaker / less item-selective at baseline `{a}` than after ≥1 analyzed item is done.
Report ‖write‖ and the peak attention onto any single task item.

In [5]:
from src.attention import get_attention_matrices

tbl = Table(title="E2b · pivot-1 vs pivot-2+ : write-norm and attention selectivity", highlight=False)
tbl.add_column("PROGRESS"); tbl.add_column("‖goal write‖", justify="right")
tbl.add_column("peak attn onto an item", justify="right"); tbl.add_column("argmax item", justify="left")
for done in [[], ["b"], ["b", "c"]]:
    p = build_prompt(done); piv = pivot_of(p)
    rich_print(Panel(Text.from_markup(exact_prompt(p, patch_pos=[piv])),
                     title=f"[bold]E2b · PROGRESS {{{','.join([ANCHOR] + done)}}} · readout @ pivot "
                           f"{piv} = {pivot_token(p, patch_pos=[piv])!r}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))
    info = get_attention_matrices(prepare_input(prompts=p, tokenizer=mt), mt, value_weighted=True)
    per_item = {k: pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum() for k in ANALYZE}
    top = max(per_item, key=per_item.get)
    tbl.add_row("{" + ",".join([ANCHOR] + done) + "}", f"{write_norm(p, GOAL_HEADS, piv):.2f}",
                f"{per_item[top]:.3f}", f"`{top}` ({'undone' if top not in done else 'DONE?!'})")
rich_print(tbl)

╭─ E2b · PROGRESS {a} · readout @ pivot 134 = ['    '] ───────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ E2b · PROGRESS {a,b} · readout @ pivot 150 = ['    '] ─────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ E2b · PROGRESS {a,b,c} · readout @ pivot 168 = ['    '] ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list of capitalized words\n                                                                             │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 E2b · pivot-1 vs pivot-2+ : write-norm and attention selectivity  
┏━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ PROGRESS ┃ ‖goal write‖ ┃ peak attn onto an item ┃ argmax item  ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ {a}      │       211.55 │                  2.286 │ `b` (undone) │
│ {a,b}    │       195.46 │                  1.821 │ `c` (undone) │
│ {a,b,c}  │       174.10 │                  2.019 │ `d` (undone) │
└──────────┴──────────────┴────────────────────────┴──────────────┘